# J2 Après-Midi | L'API Hour ou la collecte de Données Web

**Objectifs d'apprentissage :**
1. Comprendre l'intuition derrière le concept API.
2. Interroger des API existantes pour extraire des données.
3. Préparation de scripts pour automatiser la collecte de données

**Prérequis :**
- Notions basiques de Python & Pandas (J1)


## Le Web vu par les humains vs vu par les machines

Lorsque vous naviguez sur internet, votre navigateur communique avec un serveur en utilisant le protocole **HTTP**. Il récupère du texte HTML, des images, et les met en page, grâce au navigateur, pour que ce soit agréable à lire.

Mais nous pouvons aussi interagir avec les serveurs de manière plus "brute" comme le font les machines. 

Un exemple avec `curl`... 

In [ ]:
!curl "wttr.in/lille"

## 1. Qu'est-ce qu'une API ?

On entend souvent parler d'API pour extraire des données des réseaux sociaux (Twitter/X, Instagram, ...), mais aussi de Wikipedia ou des bases de données institutionnelles. N'importe quel contenu sur le web peut-etre collecté.

**API** signifie *Application Programming Interface* (Interface de Programmation d'Application).

### L'intuition - Allons au restaurant
- Vous êtes le client à la table.
- La cuisine prépare les plats.
- **L'API, c'est le serveur** : vous lui donnez votre commande (la *requête* ou *request*), il la transmet à la cuisine, et il vous rapporte votre plat (la *réponse*).


### Nous utilisons des APIs sans le savoir

On l'oublie souvent, mais les fonctions que nous utilisons dans Python (`pandas.read_csv('mes_data.csv')`, `df['var'].plot(kind='bar')`) ou dans R (`summary()`, `plot()`) sont aussi des **APIs** (ou interfaces).

Ces interfaces masquent la complexité du code source sous-jacent. Vous n'avez pas besoin de savoir *comment* `pandas` lit le fichier caractère par caractère, vous interagissez juste avec la fonction en lui passant des **arguments** (le nom du fichier).

L'API formalise les intéractions!



### Hack-Time 🛠️

Créons notre propre API ! Modifiez la cellule suivante pour créer une petite fonction Python. La manière dont vous définissez les paramètres est la **signature** de votre API locale.

In [ ]:
# Créez une fonction simple avec un ou plusieurs paramètres
def my_super_fonction():
    # À vous de jouer
    pass

# On interagit ensuite avec l'interface définie


Faisons une requête HTTP basique pour demander à une API (prend qqc de simple et facile à comprendre)

In [ ]:
import requests

# Interroger l'API ouverte du gouvernement pour obtenir des infos sur une commune
url = "https://geo.api.gouv.fr/communes?nom=Lille&fields=population"
response = requests.get(url)

# Convertir la réponse (texte brut) en un dictionnaire Python (JSON)
data = response.json()

print("Code de statut :", response.status_code)
print("Données reçues :", data)

Pour un politiste ou un sociologue, il est très d'extraire le contenu textuel.

Plutôt que de faire du scraping web complexe, nous pouvons utiliser les API ouvertes. 
Par exemple, nous pouvons, avec la bibliothèque `requests`, intéragir directement avec Wikipédia pour récupérer le texte brut d'un ou plusieurs articles.

In [ ]:
import requests

def fetch_wikipedia_page(page_title):
    """Récupère le contenu d'une page Wikipédia en utilisant l'API."""
    url = "https://fr.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "titles": page_title,
        "explaintext": True,
        "format": "json",
        "formatversion": 2
    }
    headers = {"User-Agent": "LilleLMS_SummerSchool/1.0 (https://github.com/mickaeltemporao/lillelms)"}

    response = requests.get(url, params=params, headers=headers)
    data = response.json()
    return data["query"]["pages"][0]["extract"]



In [ ]:
# Exemple avec une personnalité politique
page_title = 'Gabriel_Attal'
print(f"Téléchargement de la page : {page_title}...")
texte = fetch_wikipedia_page(page_title)
print("--- Début de l'article ---")
print(texte[:400] + "...")

### Hack-Time 🛠️

Modifiez le code ci-dessous pour : 
1. Choisir une autre page Wikipédia (ex: `Mouvement_des_Gilets_jaunes` ou le nom d'un autre politique).
2. Récupérer et afficher son texte.
3. Compter et afficher le nombre de caractères de ce texte *(Astuce : utilisez `len(texte)`)*.

**Défi optionnel** : Comptez le nombre de mots dans le texte récupéré *(Astuce : vous pouvez utiliser la méthode `.split()` sur la chaîne de caractères)*.

In [ ]:
# Modifiez le nom de la page ci-dessous
page_title = 'Gabriel_Attal'

texte = fetch_wikipedia_page(page_title)
print(f"Texte de la page {page_title} (extrait) :\n")
print(texte[:500] + "...")

### Cas d'usage : Les réseaux sociaux (API Bluesky)

Certaines plateformes sociales récentes, comme **Bluesky**, offrent des API publiques très ouvertes pour récupérer les messages postés sans même avoir besoin de s'authentifier !

Voici un exemple où nous récupérons les 5 derniers messages de 3 personnalités politiques françaises, pour créer un petit jeu de données (corpus) au format tableau :

In [ ]:
import requests
import pandas as pd

# Liste de 3 personnalités politiques sur Bluesky
politiciens = [
    "jlmelenchon.bsky.social",
    "olivierfaure.bsky.social"
]

donnees_bluesky = []

# Boucle pour interroger chaque compte
for politicien in politiciens:
    # L'URL de l'API publique avec la limite de 5 messages
    url = f"https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed?actor={politicien}&limit=5"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        
        # La structure de la réponse contient une liste 'feed' avec les messages
        for item in data.get('feed', []):
            post = item.get('post', {}).get('record', {})
            
            # On ajoute le message extrait dans notre liste
            donnees_bluesky.append({
                "Politicien": politicien,
                "Date": post.get('createdAt', ''),
                "Texte": post.get('text', '')
            })

# Création et affichage du tableau récapitulatif
df_bluesky = pd.DataFrame(donnees_bluesky)
df_bluesky

## 3. Parfois il n'y a pas d'API ... 

Que faire lorsqu'un site web ne propose pas d'API ? Il faut aller chercher l'information directement dans le code source de la page (HTML). C'est le **Web Scraping**.

Nous allons interroger la page web avec `requests` et utiliser `BeautifulSoup` pour lire la structure HTML et extraire les données.

**Attention:**
- Avant de lancer une collecte de données à grande échelle, sans passer par une API offcielle, il faut mieux contacter l'administrateur du site qui très souvent lrosque vous explqieur votre projet vous donnera les donnés.
Donc pour réuser 
etape 1 on check s'il y a une api 
etape 2 si pas, on contact pour informer qu'on va collecter des donnes pour un porjet de recherche
etabe 3 si ok on lance la collece 


In [ ]:
# Exemple simple d'introduction au Web Scraping
from bs4 import BeautifulSoup
import requests

url = "https://www.mediapart.fr/"
response = requests.get(url)

# On analyse le code HTML de la page avec BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

# On cherche le titre principal de la page (la balise <h1>)
title = soup.find("h1").get_text()

print("Le titre de la page est :", title)

Imaginons que nous voulions étudier le temps d'antenne consacré aux "Zones à Faibles Émissions" (ZFE) sur différentes chaînes d'information (TF1, CNews). Le catalogue de l'INA (Institut National de l'Audiovisuel) propose un moteur de recherche.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

# URL du catalogue de l'INA
base_url = "https://catalogue.ina.fr/docListe/TV-RADIO"

# On reproduit les filtres de la barre de recherche
params_ina = {
    "base_label": "TVNAT,TVSAT",  # TV Nationale et Satellite
    "itoustext": "ZFE",           # Notre mot clé
    "ch": "CNews",                # La chaîne
    "nbLignes": 10                # Nombre de résultats par page
}

# On se présente pour éviter d'être bloqué par le site
headers = {'User-Agent': 'LilleLMS_SummerSchool/1.0 (https://github.com/mickaeltemporao/lillelms)'}

# 1. Récupération du code HTML
print("Téléchargement de la page de l'INA...")
response_ina = requests.get(base_url, params=params_ina, headers=headers)
html_content = response_ina.text

# 2. Parsing (Analyse de la structure) avec BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

# Dans le code HTML de l'INA, les résultats sont dans des lignes de tableau <tr> 
# ayant une classe qui contient 'result_line'.
rows = soup.find_all("tr", class_=lambda x: x and "result_line" in x)
print(f"Nombre de résultats trouvés sur cette page : {len(rows)}")

Dans la boucle suivante, nous allons inspecter chaque ligne (`row`) et extraire le texte des colonnes (`td`).

In [ ]:
ina_data = []

for row in rows:
    cells = row.find_all("td")
    if len(cells) >= 9:
        # L'indexation commence à 0. Dans ce tableau:
        # cells[1] = Chaîne, cells[2] = Date, cells[4] = Durée, cells[5] = Titre
        
        chaine = cells[1].get_text(strip=True)
        date = cells[2].get_text(strip=True)
        titre = cells[5].get_text(strip=True)
        
        ina_data.append({
            "chaine": chaine,
            "date": date,
            "titre": titre
        })

# On transforme notre liste en un tableau propre (DataFrame pandas)
df_ina = pd.DataFrame(ina_data)
display(df_ina.head())

### Hack-Time 🛠️

Observez comment le `titre` et la `date` ont été extraits en utilisant l'indice des cellules HTML (`cells[1]`, `cells[2]`, etc.).

Sachant que la colonne **"Durée"** se trouve à la 5ème place dans le tableau HTML de l'INA, modifiez le code ci-dessus pour ajouter également la durée de l'émission dans votre `DataFrame`.

## 4. IA & API

Jusqu'à présent, nous avons utilisé des APIs pour collecter des données existantes (Wikipédia, INA). Mais nous pouvons aussi utiliser les des API pour interroger des modèles de langage pour générer ou analyser du texte.

### 4.1. L'API OpenAI

Pour interroger des modèles comme GPT-3.5 ou GPT-4, nous utilisons l'API officielle d'OpenAI. C'est un service payant qui nécessite une clé d'authentification (`API_KEY`). **Ne partagez jamais cette clé publiquement !**

In [ ]:
import os
from openai import OpenAI

# 1. Coller votre clé API d'OPEN AI
api_key = "sk-votre-cle-secrete-ici"

# 2. On initialise le "client" qui va communiquer avec l'API
client = OpenAI(api_key=api_key)

def demander_a_gpt(prompt):
    """Envoie un prompt à OpenAI et retourne la réponse."""
    try:
        response = client.chat.completions.create(
            model="gpt-5.4-nano",
            messages=[
                {
                    "role": "user", 
                    "content": prompt
                }
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erreur de connexion : {e}"

# 3. ... utilisez votre fonction. mais en avons nous besoin? 

### 4.2. L'alternative HuggingFace : Modèles locaux sans clé API

HuggingFace est une plateforme communautaire hébergeant des milliers de modèles ouverts (*Open Source*). Au lieu de passer par une API distante (qui nécessite souvent de s'authentifier), nous pouvons télécharger directement un modèle et l'exécuter localement !

Pour cela, nous allons utiliser la librairie `transformers` et sa fonction `pipeline`. C'est une excellente illustration de ce que nous avons vu plus tôt : le `pipeline` agit comme une **API locale**, qui masque toute la complexité du réseau de neurones sous-jacent.

In [ ]:
from transformers import pipeline

# 1. On charge l'interface (pipeline) avec un modèle d'analyse de sentiment en français
# Le modèle sera téléchargé automatiquement la première fois
analyseur = pipeline("sentiment-analysis", model="cmarkea/distilcamembert-base-sentiment")

# 2. On interroge notre modèle local
texte = "Les modèles locaux sont pratiques car nos données restent protégés localement!" 
resultat = analyseur(texte)

print("Résultat de l'analyse :", resultat)

## Conclusion

Dans un projet de recherche, il faut souvent exécuter le même type de boucle sur des centaines de pages pour construire un corpus. **Cependant**, faire trop de requêtes web à la suite peut surcharger les serveurs... Ce qui peut bloquer notre adresse IP, avec raison!

Pour la suite de l'atelier, nous allons parfois utiliser **des jeux de données pré-collectés** qui contiennent les données collectés pour éviter de faire tomber des serveurs...